In [ ]:
%%writefile crop_tool.py

import os
import sys
import zipfile
import tempfile
import shutil
from PIL import Image
from tqdm import tqdm

# Suppress DecompressionBombError for very large images. Use with caution.
Image.MAX_IMAGE_PIXELS = None

def robust_image_crop(input_zip_path, output_dir):
    """
    A reliable, file-based script to crop images from a zip file based on a 3x3 grid.

    Workflow:
    1. Create a temporary directory.
    2. Extract the input zip to this temp directory.
    3. Create output subdirectories for the two new crop types.
    4. Find images, crop them to the top-right and middle-right grid cells, and save them.
    5. Zip up the output subdirectories.
    6. Clean up all temporary files.
    """
    if not os.path.exists(input_zip_path):
        print(f"❌ ERROR: Input file not found at '{input_zip_path}'")
        return

    # --- 1. Create a master temporary directory for all our work ---
    with tempfile.TemporaryDirectory() as temp_dir:
        print(f"Created temporary directory: {temp_dir}")

        # Define paths for our operations
        extract_path = os.path.join(temp_dir, 'extracted')
        # --- NEW: Updated paths for the new crop types ---
        top_right_grid_path = os.path.join(temp_dir, 'grid_top_right')
        middle_right_grid_path = os.path.join(temp_dir, 'grid_middle_right')

        # --- 2. Extract the input zip file ---
        try:
            print(f"Extracting '{os.path.basename(input_zip_path)}'...")
            with zipfile.ZipFile(input_zip_path, 'r') as zip_ref:
                zip_ref.extractall(extract_path)
        except zipfile.BadZipFile:
            print(f"❌ ERROR: The file '{input_zip_path}' is not a valid zip file.")
            return
        except Exception as e:
            print(f"❌ ERROR: Failed to extract zip file. Reason: {e}")
            return

        # --- 3. Create output subdirectories ---
        os.makedirs(top_right_grid_path, exist_ok=True)
        os.makedirs(middle_right_grid_path, exist_ok=True)

        # --- 4. Find and process all image files ---
        image_files_to_process = []
        for root, _, files in os.walk(extract_path):
            for file in files:
                if file.lower().endswith(('.png', '.jpg', '.jpeg', '.bmp', '.gif', '.tif', '.tiff')):
                    image_files_to_process.append(os.path.join(root, file))

        if not image_files_to_process:
            print("⚠️ WARNING: No image files found in the zip archive.")
            return

        print(f"\nFound {len(image_files_to_process)} images to process.")
        for img_path in tqdm(image_files_to_process, desc="Processing images"):
            try:
                with Image.open(img_path) as img:
                    # Ensure image is in a standard format to avoid errors
                    img_rgb = img.convert("RGB")
                    width, height = img_rgb.size
                    base_filename = os.path.basename(img_path)
                    
                    # --- NEW: 3x3 GRID CROPPING LOGIC ---
                    
                    # Crop Top-Right Grid Cell (Row 1, Column 3)
                    # Coords are (left, upper, right, lower)
                    tr_coords = (int(2 * width / 3), 0, width, int(height / 3))
                    cropped_tr = img_rgb.crop(tr_coords)
                    cropped_tr.save(os.path.join(top_right_grid_path, f"tr_grid_{base_filename}"), "JPEG")

                    # Crop Middle-Right Grid Cell (Row 2, Column 3)
                    mr_coords = (int(2 * width / 3), int(height / 3), width, int(2 * height / 3))
                    cropped_mr = img_rgb.crop(mr_coords)
                    cropped_mr.save(os.path.join(middle_right_grid_path, f"mr_grid_{base_filename}"), "JPEG")

            except Exception as e:
                print(f"\n⚠️ SKIPPED: Could not process '{os.path.basename(img_path)}'. Reason: {e}")

        # --- 5. Zip up the results ---
        print("\nArchiving results...")
        os.makedirs(output_dir, exist_ok=True) # Ensure final output directory exists

        # --- NEW: Updated final zip filenames ---
        top_right_zip_final_path = os.path.join(output_dir, 'cropped_top_right_grid.zip')
        middle_right_zip_final_path = os.path.join(output_dir, 'cropped_middle_right_grid.zip')

        shutil.make_archive(top_right_zip_final_path.replace('.zip', ''), 'zip', top_right_grid_path)
        shutil.make_archive(middle_right_zip_final_path.replace('.zip', ''), 'zip', middle_right_grid_path)

        # --- 6. Final confirmation ---
        print("\n✅ Success! Processing complete.")
        print(f"Files saved to: {os.path.abspath(output_dir)}")
        print(f"  - {top_right_zip_final_path}")
        print(f"  - {middle_right_zip_final_path}")

if __name__ == "__main__":
    if len(sys.argv) != 3:
        print("Usage: python crop_tool.py <path_to_input_zip> <path_to_output_directory>")
        sys.exit(1)

    input_zip = sys.argv[1]
    output_dir = sys.argv[2]
    robust_image_crop(input_zip, output_dir)

In [ ]:
# @title 1. Configuration
# --- IMPORTANT: EDIT THESE TWO LINES ---

# The full path to the zip file you want to process.
# Example Linux/Colab: "/content/my_images.zip"
# Example Windows: "C:/Users/User/Desktop/my_images.zip"
input_zip_file = "/content/images.zip"

# The full path to the folder where you want to save the final results.
# The script will create this folder if it doesn't exist.
# Example Linux/Colab: "/content/final_output"
# Example Windows: "C:/Users/User/Desktop/final_output"
output_folder = "/content/processed_output"

# ----------------------------------------

print("Configuration set.")
print(f"Input file: {input_zip_file}")
print(f"Output folder: {output_folder}")

In [ ]:
# @title 2. Run Processing
# This cell executes the robust script we created.
import sys

# We construct the command-line instruction here
# It will look like: python crop_tool.py "/path/to/input.zip" "/path/to/output"
command = f'python crop_tool.py "{input_zip_file}" "{output_folder}"'

# Execute the command
!{command}